## Final Preprocessing – SMILES to Molecular Graphs

This notebook converts the final curated datasets into molecular graph representations for graph neural network training across three learning stages.

The processed datasets are:

1. **Unified single-drug toxicity dataset**
   - used for Phase 2 single-drug encoder pretraining
   - converts each SMILES into a PyTorch Geometric graph
   - attaches pLD50 as the regression target
   - also saves a standardized version of the target

2. **MUDI train and test datasets**
   - used for Phase 3 interaction classification
   - converts each drug pair into two graph objects
   - stores the corresponding class labels
   - computes class weights to reduce class imbalance during training

3. **Unified mixture toxicity dataset**
   - used for Phase 4 fine-tuning for mixture LD50 regression
   - converts each drug pair into two graph objects
   - stores mixture toxicity targets in −log10(mol/kg)
   - also saves a standardized target version for stable regression training

The final outputs are saved as `.pt` files so graph construction does not need to be repeated during model training.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install rdkit
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 44.7 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.0.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.7 MB/s eta 0:00:00
  Created wheel for torch-scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=677289 sha256=55d3471a6614b15080290fd3704d652e20a50dee8a6cf3c4741237628e0d162b
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
  Create

In [3]:
import os
import joblib
import torch
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import Descriptors

from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

RDLogger.DisableLog('rdApp.*')

In [4]:
base_path = '/content/drive/MyDrive/FYP/IRP/Data'
output_dir = os.path.join(base_path, 'processed_graphs')
os.makedirs(output_dir, exist_ok=True)

single_csv = os.path.join(base_path, 'unified_single_drug_pld50.csv')
mixture_csv = os.path.join(base_path, 'unified_mixture_toxicity_model_ready.csv')
mudi_train_csv = os.path.join(base_path, 'MUDI', 'train_processed.csv')
mudi_test_csv = os.path.join(base_path, 'MUDI', 'test_processed.csv')

print("Single-drug dataset:", single_csv)
print("Unified mixture dataset:", mixture_csv)
print("MUDI train dataset:", mudi_train_csv)
print("MUDI test dataset:", mudi_test_csv)

Single-drug dataset: /content/drive/MyDrive/FYP/IRP/Data/unified_single_drug_pld50.csv
Unified mixture dataset: /content/drive/MyDrive/FYP/IRP/Data/unified_mixture_toxicity_model_ready.csv
MUDI train dataset: /content/drive/MyDrive/FYP/IRP/Data/MUDI/train_processed.csv
MUDI test dataset: /content/drive/MyDrive/FYP/IRP/Data/MUDI/test_processed.csv


## Atom and bond feature functions

In [5]:
def atom_features(atom):
    """
    Extract a feature vector for an atom.
    Features:
      - Atom symbol one-hot: C, N, O, S, F, P, Cl, Br, I, B
      - Degree one-hot: 0 to 6
      - Formal charge (continuous, will be scaled later)
      - Hybridization: sp, sp2, sp3
      - Aromaticity (binary)
      - Atomic mass (continuous)
      - Number of hydrogens (continuous)
    """

    # Atom Symbol (What element is it?)
    # One-hot for atom symbol (common elements)
    symbol = atom.GetSymbol()
    symbols = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br', 'I', 'B']
    symbol_vec = [1.0 if symbol == s else 0.0 for s in symbols]

    # Degree (How many atoms are connected to it?)
    # Degree (up to 6)
    degree = atom.GetDegree()
    degree_vec = [0.0]*7
    degree_vec[min(degree, 6)] = 1.0

    # Formal Charge (Electrical charge)
    formal_charge = atom.GetFormalCharge()

    # Hybridization (Shape of electron orbitals)
    '''This changes molecule stability and reactions.'''
    hyb = atom.GetHybridization()
    hyb_vec = [
        1.0 if hyb == Chem.rdchem.HybridizationType.SP else 0.0,
        1.0 if hyb == Chem.rdchem.HybridizationType.SP2 else 0.0,
        1.0 if hyb == Chem.rdchem.HybridizationType.SP3 else 0.0
    ]

    # 5) Aromaticity (Special stable ring?)
    '''Aromatic atoms belong to special stable rings (like benzene).
    These rings strongly affect drug behavior and toxicity.'''
    is_aromatic = 1.0 if atom.GetIsAromatic() else 0.0

    # Atomic mass
    '''Heavier atoms often increase toxicity and bioactivity'''
    mass = atom.GetMass()

    # Number of hydrogens (implicit + explicit)
    '''Number of hydrogens affects solubility and reactivity'''
    num_h = atom.GetTotalNumHs()

    features = symbol_vec + degree_vec + [formal_charge] + hyb_vec + [is_aromatic, mass, num_h]
    return np.array(features, dtype=np.float32)

def bond_features(bond):
    """
    Extract features for a bond.
    Features:
      - Bond type one-hot: single, double, triple, aromatic
      - Conjugation (binary)
      - In ring (binary)
    """
    bond_type = bond.GetBondType() # Bond Type (strength of connection)
    '''Bond type changes: molecule stability, reactivity, toxicity'''
    bond_type_vec = [
        1.0 if bond_type == Chem.rdchem.BondType.SINGLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.DOUBLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.TRIPLE else 0.0,
        1.0 if bond_type == Chem.rdchem.BondType.AROMATIC else 0.0

    ]
    is_conjugated = 1.0 if bond.GetIsConjugated() else 0.0
    '''Conjugation means, Electrons can move across multiple atoms (alternating double bonds).
       Important cuz increases chemical reactivity, affects drug absorption, affects toxicity strongly.
       Many toxic compounds are conjugated systems.'''

    is_in_ring = 1.0 if bond.IsInRing() else 0.0
    '''Checks whether bond belongs to a ring structure.
       Rings often: stabilize molecules, change metabolism, increase persistence in body.
       Many drugs and toxins contain rings.'''
    return np.array(bond_type_vec + [is_conjugated, is_in_ring], dtype=np.float32)

These feature definitions follow standard practice in molecular GNNs (e.g., Gilmer et al., 2017; Yang et al., 2019). Including atomic properties like symbol, degree, hybridization, and aromaticity captures essential chemical information, while bond type and conjugation (electrons are shared across multiple adjacent bonds) help the model understand connectivity and electronic effects.

Hybridization (Shape of electron orbitals)
| Type | Shape  | Example |
| ---- | ------ | ------- |
| sp   | linear | CO₂     |
| sp2  | flat   | benzene |
| sp3  | 3D     | methane |


## SMILES to graph conversion

In [6]:
def smiles_to_graph(smiles):
    """
    Convert a SMILES string to a PyTorch Geometric Data object.
    Returns None if SMILES is invalid.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # Atom features (nodes)
        atom_feats = [atom_features(atom) for atom in mol.GetAtoms()]
        x = torch.tensor(np.array(atom_feats), dtype=torch.float)

        # Bond features (edges)
        edge_indices = []
        edge_feats = []

        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            edge_indices += [[i, j], [j, i]]
            feats = bond_features(bond)
            edge_feats.append(feats)
            edge_feats.append(feats)

        if len(edge_indices) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr = torch.empty((0, 6), dtype=torch.float)
        else:
            edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(np.array(edge_feats), dtype=torch.float)

        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
        return data

    except Exception:
        return None

This function transforms a SMILES string into a graph representation that PyTorch Geometric can consume. It creates node features (x), edge indices (connectivity), and edge features (edge_attr). The graph is undirected, so each bond is stored twice (both directions).

## Helper functions for processing datasets

In [7]:
def process_single_drug_dataset(csv_path, smiles_col, target_col, output_graph_file, output_target_file):
    df = pd.read_csv(csv_path)

    graph_data = []
    targets = []
    failed = 0

    for _, row in df.iterrows():
        smiles = row[smiles_col]
        target = row[target_col]

        data = smiles_to_graph(smiles)
        if data is None:
            failed += 1
            continue

        data.y = torch.tensor([target], dtype=torch.float)
        graph_data.append(data)
        targets.append(target)

    print(f"{os.path.basename(csv_path)}")
    print(f"Processed: {len(graph_data)}")
    print(f"Failed: {failed}")

    torch.save(graph_data, os.path.join(output_dir, output_graph_file))
    torch.save(torch.tensor(targets, dtype=torch.float), os.path.join(output_dir, output_target_file))

    return graph_data, targets


def process_pair_regression_dataset(csv_path, smiles_a_col, smiles_b_col, target_col, output_file):
    df = pd.read_csv(csv_path)

    graphs_a = []
    graphs_b = []
    targets = []
    failed = 0

    for _, row in df.iterrows():
        smiles_a = row[smiles_a_col]
        smiles_b = row[smiles_b_col]
        target = row[target_col]

        data_a = smiles_to_graph(smiles_a)
        data_b = smiles_to_graph(smiles_b)

        if data_a is None or data_b is None:
            failed += 1
            continue

        graphs_a.append(data_a)
        graphs_b.append(data_b)
        targets.append(target)

    print(f"{os.path.basename(csv_path)}")
    print(f"Processed: {len(targets)}")
    print(f"Failed: {failed}")

    torch.save(
        (graphs_a, graphs_b, torch.tensor(targets, dtype=torch.float)),
        os.path.join(output_dir, output_file)
    )

    return graphs_a, graphs_b, targets


def process_pair_classification_dataset(csv_path, smiles_a_col, smiles_b_col, label_col, output_file):
    df = pd.read_csv(csv_path)

    graphs_a = []
    graphs_b = []
    labels = []
    failed = 0

    for _, row in df.iterrows():
        smiles_a = row[smiles_a_col]
        smiles_b = row[smiles_b_col]
        label = row[label_col]

        data_a = smiles_to_graph(smiles_a)
        data_b = smiles_to_graph(smiles_b)

        if data_a is None or data_b is None:
            failed += 1
            continue

        graphs_a.append(data_a)
        graphs_b.append(data_b)
        labels.append(label)

    print(f"{os.path.basename(csv_path)}")
    print(f"Processed: {len(labels)}")
    print(f"Failed: {failed}")

    torch.save(
        (graphs_a, graphs_b, torch.tensor(labels, dtype=torch.long)),
        os.path.join(output_dir, output_file)
    )

    return graphs_a, graphs_b, labels

## Process Unified Single‑Drug Dataset


In [8]:
single_graphs, single_targets = process_single_drug_dataset(
    csv_path=single_csv,
    smiles_col='SMILES',
    target_col='pLD50',
    output_graph_file='unified_single_drug.pt',
    output_target_file='unified_single_targets.pt'
)

unified_single_drug_pld50.csv
Processed: 13579
Failed: 0


## Standardize single-drug targets for pretraining

In [9]:
single_targets_np = np.array(single_targets).reshape(-1, 1)

single_scaler = StandardScaler()
single_scaler.fit(single_targets_np)
single_targets_scaled = single_scaler.transform(single_targets_np)

joblib.dump(single_scaler, os.path.join(output_dir, 'single_pld50_scaler.pkl'))

for i, data in enumerate(single_graphs):
    data.y = torch.tensor(single_targets_scaled[i], dtype=torch.float)

torch.save(single_graphs, os.path.join(output_dir, 'unified_single_drug_scaled.pt'))

print("Saved scaled single-drug graphs and scaler.")

Saved scaled single-drug graphs and scaler.


## Process MUDI Dataset (Interaction Learning)

In [10]:
mudi_train_graphs_a, mudi_train_graphs_b, labels_train = process_pair_classification_dataset(
    csv_path=mudi_train_csv,
    smiles_a_col='smiles_a',
    smiles_b_col='smiles_b',
    label_col='label',
    output_file='mudi_train.pt'
)

mudi_test_graphs_a, mudi_test_graphs_b, labels_test = process_pair_classification_dataset(
    csv_path=mudi_test_csv,
    smiles_a_col='smiles_a',
    smiles_b_col='smiles_b',
    label_col='label',
    output_file='mudi_test.pt'
)

train_processed.csv
Processed: 175045
Failed: 0
test_processed.csv
Processed: 69876
Failed: 0


## Save MUDI class weights for later training

In [11]:
# original classes and weights
classes = np.array([0, 1, 2])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=np.array(labels_train))

# remove class 2
mask = classes != 2
classes = classes[mask]
weights = weights[mask]

# convert to tensor
weights = torch.tensor(weights, dtype=torch.float)

print("Filtered class weights:", {int(c): float(w) for c, w in zip(classes, weights)})

torch.save(weights, os.path.join(output_dir, 'mudi_class_weights.pt'))

Filtered class weights: {0: 0.416185200214386, 1: 2.1357369422912598}


So rare class → large weight

common class → small weight

## Process unified mixture toxicity dataset (Fine‑tuning)

In [12]:
mix_graphs_a, mix_graphs_b, mix_targets = process_pair_regression_dataset(
    csv_path=mixture_csv,
    smiles_a_col='SMILES_A',
    smiles_b_col='SMILES_B',
    target_col='Target_neglog_molkg',
    output_file='unified_mixture_toxicity.pt'
)

unified_mixture_toxicity_model_ready.csv
Processed: 387
Failed: 0


## Standardize unified mixture regression targets

In [13]:
mix_targets_np = np.array(mix_targets).reshape(-1, 1)

mixture_scaler = StandardScaler()
mixture_scaler.fit(mix_targets_np)
mix_targets_scaled = mixture_scaler.transform(mix_targets_np)

joblib.dump(mixture_scaler, os.path.join(output_dir, 'mixture_toxicity_scaler.pkl'))

torch.save(
    (
        mix_graphs_a,
        mix_graphs_b,
        torch.tensor(mix_targets_scaled, dtype=torch.float)
    ),
    os.path.join(output_dir, 'unified_mixture_toxicity_scaled.pt')
)

print("Saved scaled mixture graphs and scaler.")

Saved scaled mixture graphs and scaler.


## Summary of generated files

In [14]:
generated_files = sorted(os.listdir(output_dir))
print("Generated files:")
for f in generated_files:
    print("-", f)

Generated files:
- best_ddi_encoder.pt
- best_ddi_model.pt
- best_ddi_reg.pt
- best_ddi_reg_encoder.pt
- best_ddi_reg_model.pt
- best_encoder.pt
- best_single_drug_model.pt
- mixture_toxicity_scaler.pkl
- mudi_class_weights.pt
- mudi_test.pt
- mudi_train.pt
- phase2_model_comparison.csv
- phase3_model_comparison.csv
- phase4_model_comparison.csv
- phase5_benchmark_outputs
- phase5_selected_explanations.csv
- phase5_target_pair_explanation.txt
- phase5_xai_method_comparison.csv
- pld50_scaler.pkl
- pretrained_encoder.pt
- single_pld50_scaler.pkl
- smyth_ddi.pt
- unified_mixture_toxicity.pt
- unified_mixture_toxicity_scaled.pt
- unified_single_drug.pt
- unified_single_drug_scaled.pt
- unified_single_targets.pt
- unified_targets.pt
